# 4. Model - Baseline MLP bằng PyTorch

> Chạy notebook từ thư mục project hoặc để cell đầu tự dò `config.json` ở thư mục cha.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.json").exists())
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

Project root: D:\setup\BTL01_Nhom_HoTen_MSSV\BTL01_Nhom_HoTen_MSSV


## 4.1 Cấu hình baseline
Baseline dùng 2 đầu vào, 1 hidden layer 10 neuron, tanh cho hidden layer và đầu ra tuyến tính cho hồi quy. Adam, learning rate 0.02, batch size 256; ngân sách tìm kiếm tối đa 100 epochs, patience 20.

In [2]:
import json, pandas as pd
from src.mlp_core import MLPRegressor, train_model
cfg=json.loads((ROOT/"config.json").read_text())
print(cfg)
model=MLPRegressor(2,[10]); print(model)

{'project_title': 'Function Approximation with Multi-Layer Perceptron', 'input_dim': 2, 'output_dim': 1, 'mu': [0.0, 0.0], 'covariance': [[0.3, -0.5], [-0.5, 2.0]], 'x1_range': [-2.0, 2.0], 'x2_range': [-4.0, 4.0], 'n_train': 2000, 'n_validation': 500, 'n_test': 100, 'data_seeds': {'train': 2026, 'validation': 2027, 'test': 2028}, 'experiment_seeds': [42, 123, 2026], 'baseline_hidden_dims': [10], 'batch_size': 256, 'epochs': 100, 'final_refit_epochs': 400, 'final_refit_patience': 40, 'learning_rate': 0.02, 'optimizer': 'Adam', 'activation': 'Tanh hidden layers + linear output', 'weight_decay': 1e-05, 'early_stopping_patience': 20, 'experiment_A_train_sizes': [200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000], 'experiment_B_hidden_widths': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 'experiment_C_hidden_depths': [1, 2, 3, 4], 'device': 'auto'}
MLPRegressor(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=10, bias=True)
    (1): Tanh()
    (2): Linear(in_features=10, out_

## 4.2 Huấn luyện có validation và checkpoint
Vòng lặp được triển khai tường minh trong `src/mlp_core.py`: `model.train()` → forward → MSE → `zero_grad()` → `backward()` → `optimizer.step()`. Validation chạy trong `model.eval()` và `torch.no_grad()`. Checkpoint tốt nhất được quyết định bằng validation MSE.

In [3]:
tr=pd.read_csv(ROOT/"data/train.csv"); va=pd.read_csv(ROOT/"data/validation.csv")
res=train_model(tr[["x1","x2"]].to_numpy(dtype="float32"), tr.y.to_numpy(dtype="float32"),
                va[["x1","x2"]].to_numpy(dtype="float32"), va.y.to_numpy(dtype="float32"),
                hidden_dims=[10], seed=42, epochs=cfg["epochs"], batch_size=cfg["batch_size"],
                learning_rate=cfg["learning_rate"], patience=cfg["early_stopping_patience"], device="cpu")
print({k:res[k] for k in ["best_epoch","train_mse","val_mse","val_mae","train_seconds"]})

{'best_epoch': 99, 'train_mse': 0.0004745504930615425, 'val_mse': 0.00046033085882663725, 'val_mae': 0.01673684024810791, 'train_seconds': 2.2563484999991488}
